<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
import os

if not os.path.exists('FlyRank_AI'):
    !git clone -q https://github.com/AhmedMahmoud-123/FlyRank_AI.git

os.chdir('FlyRank_AI')

!python scripts/01_prepare_features.py

import sys
sys.path.append('scripts')

import json
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(
    'data/processed/refresh_feature_vector.csv'
)

print(f'{len(df):,} rows loaded')

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/FlyRank_AI/FlyRank_AI/data/processed/refresh_feature_vector.csv
30,000 rows loaded


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [20]:
from sklearn.model_selection import GroupKFold, cross_val_predict

feature_cols = [
    'impressions_prev_30d',
    'avg_position',
    'days_since_last_update',
    'has_clicks'
]

model_data = df.dropna(
    subset=feature_cols + [
        'is_declining_label',
        'client_id'
    ]
).copy()

X = model_data[feature_cols]
y = model_data['is_declining_label']
groups = model_data['client_id']

# Confirm the target is binary before using predict_proba()[:, 1].
target_values = sorted(y.dropna().unique())

assert target_values == [0, 1], (
    f'Expected binary target [0, 1], found {target_values}'
)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

gkf = GroupKFold(n_splits=5)

oof_probs = cross_val_predict(
    rf,
    X,
    y,
    groups=groups,
    cv=gkf,
    method='predict_proba',
    n_jobs=-1
)

model_data['risk_score'] = oof_probs[:, 1]

median_pos = model_data.loc[
    model_data['avg_position'] > 0,
    'avg_position'
].median()

def reason_code(row):
    if (
        row['days_since_last_update'] >= 180
        and row['impressions_prev_30d'] >= 500
    ):
        return 'stale_but_visible'

    if row['avg_position'] > median_pos:
        return 'weak_position'

    return 'low_signal'

model_data['reason_code'] = model_data.apply(
    reason_code,
    axis=1
)

queue = model_data.sort_values(
    'risk_score',
    ascending=False
).copy()

base_rate = y.mean()

print(
    f'base rate: {base_rate:.3f} ({base_rate:.1%})'
)

queue[
    [
        'content_id',
        'risk_score',
        'reason_code',
        'impressions_prev_30d',
        'avg_position',
        'days_since_last_update'
    ]
].head(20)

base rate: 0.542 (54.2%)


,content_id,risk_score,reason_code,impressions_prev_30d,avg_position,days_since_last_update
23624,content_0a90dfb2229d,1.0,weak_position,13,12.1,8
1949,content_21963a7d58d2,1.0,weak_position,26566,27.0,20
18360,content_899a6bb90034,1.0,low_signal,7,7.6,8
22605,content_9bcc697ea7f3,1.0,low_signal,351,7.1,20
470,content_d0380d2bc615,1.0,weak_position,347,15.0,104
19298,content_f87ff9f55f70,1.0,weak_position,1041,18.1,14
16210,content_3d1c277aa7b9,1.0,low_signal,1,5.0,104
8468,content_26e7e98c8e41,1.0,low_signal,338,2.9,104
713,content_624e59919b5e,1.0,weak_position,257,21.5,104
7702,content_fd40fc33802e,1.0,weak_position,3,15.8,8


**Ranked actions:** each row is scored by predicted decline risk, with a plain-language reason
code attached — `stale_but_visible` (real traffic, untouched a long time), `weak_position`
(worse-than-median average position), or `low_signal` (very little recent traffic). A person
reading row 1 should understand *why* it is prioritized without reading any code, per the
baseline's transparency standard from Week 4.

The `weak_position` label describes the current position relative to the dataset median; it
does not imply that the page's position has deteriorated over time.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
n_clients = groups.nunique()

print(
    f"Queue covers {len(queue):,} content items from {n_clients} clients. "
    f"Risk scores are out-of-fold predictions from {gkf.get_n_splits()}-fold "
    f"grouped cross-validation (GroupKFold on client_id) -- every score reflects "
    f"a model that never saw that content item's client during training."
)

Queue covers 30,000 content items from 32 clients. Risk scores are out-of-fold predictions from 5-fold grouped cross-validation (GroupKFold on client_id) -- every score reflects a model that never saw that content item's client during training.


**Intended use:** a content strategist's starting point for weekly review — which pages to look at first, not a final verdict. Built for the Refresh / Content Opportunity lane specifically; not validated for other content types or other clients' data outside this dataset.

**Limits (claim-ladder honest):**

- This is a **decision-support ranking**, not a prediction of what will happen to any single page — cross-sectional data from one dataset snapshot cannot support "refreshing this page will fix it."
- Risk scores are out-of-fold predictions from client-grouped cross-validation. The separate validation audit in W06 provides the primary held-out evaluation used to assess whether the ranking generalizes beyond the training clients.
- The leakage audit (W03/W06) removed features that inflated earlier accuracy numbers — this queue's risk scores are the post-audit version, not the higher pre-audit numbers.
- The observed target base rate in this dataset is reported above. A risk score should be interpreted relative to that baseline; a score only slightly above the base rate is not necessarily a strong signal.
- This ranking has not been validated on a genuinely new client's data outside this dataset.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [21]:
visible = (
    model_data['impressions_prev_30d'] >= 500
).astype(int)

stale = (
    model_data['days_since_last_update'] >= 180
).astype(int)

median_pos = model_data.loc[
    model_data['avg_position'] > 0,
    'avg_position'
].median()

weak_position = (
    model_data['avg_position'] > median_pos
).astype(int)

baseline_flag = (
    visible & (stale | weak_position)
).astype(int)


comparison_threshold = 0.5

model_flag = (
    model_data['risk_score'] >= comparison_threshold
).astype(int)

disagreement = model_data[
    baseline_flag != model_flag
]

print(
    f'{len(disagreement):,} rows where model and baseline '
    f'rule disagree at risk_score >= {comparison_threshold:.1f} '
    f'-- review these first'
)

disagreement[
    [
        'content_id',
        'risk_score',
        'reason_code'
    ]
].head(10)

15,481 rows where model and baseline rule disagree at risk_score >= 0.5 -- review these first


,content_id,risk_score,reason_code
0,content_304f48230142,0.74000,low_signal
5,content_d4084a4bc775,0.76000,low_signal
6,content_9a34b442b552,0.79281,low_signal
7,content_a63219c6e95a,0.47500,weak_position
9,content_c27558df2b0c,0.69500,low_signal
12,content_42fb2cad9ecf,0.60000,low_signal
13,content_a5a2fbc76336,0.83000,weak_position
14,content_91067a14431a,0.74500,weak_position
15,content_689414059706,0.98000,low_signal
16,content_78bd1d4a1d4d,0.58500,low_signal


In [22]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top_k = order[:k]
    return np.asarray(y_true)[top_k].mean()


precision_results = []

for k in [20, 50, 100, 200]:
    precision_results.append({
        'k': k,
        'precision_at_k': precision_at_k(
            y,
            model_data['risk_score'],
            k
        )
    })

precision_results = pd.DataFrame(
    precision_results
)

print('Pooled OOF Precision@K:')
display(precision_results)

Pooled OOF Precision@K:


,k,precision_at_k
0,20,0.950
1,50,0.920
2,100,0.940
3,200,0.855


**What a person must check before acting:**
- Rows where the model and Week-4 baseline rule disagree (above) — either could be right;
  a human should look at the actual page, not just trust the higher-tech option by default.
- Any row with a reason code but very low `impressions_prev_30d` (near-zero traffic) — the
  model may be picking up noise on a page nobody's really looking at anyway.

**No-go list — never automate:**
- Do not auto-publish or auto-edit content based on this score alone; it flags candidates for
  a human writer/editor, nothing more.
- Do not use this ranking to make client-facing promises about traffic outcomes — no causal
  claim is supported (per writing-honest-claims: cross-sectional data, no intervention design).
- Do not apply this model to a client or content type outside the Refresh lane without
  re-validating — the honest-split numbers here are specific to this data.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signs this playbook has gone stale:**

- If the gap between random-split and grouped-split performance, as measured in W06, widens after future data refreshes, that would suggest the model may be relying more heavily on client-specific patterns.
- If the base rate of `is_declining_label` shifts meaningfully from the measured dataset base rate, the risk scores may need recalibration rather than being reused unchanged.
- If reason codes increasingly fail to match what reviewers find during inspection, the disagreement log should be investigated before continuing to rely on the ranking.
- **Retrain / re-audit trigger:** re-run W06's leakage-and-split audit whenever the raw data is refreshed with a new month of `content_refresh_anonymized.csv`, not only when performance appears to decline. Leakage can return if the feature-preparation script changes.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [23]:
Path('work/outputs').mkdir(
    parents=True,
    exist_ok=True
)

# Action queue: information available when making the ranking.
queue_cols = [
    'content_id',
    'risk_score',
    'reason_code',
    'impressions_prev_30d',
    'avg_position',
    'days_since_last_update'
]

queue[queue_cols].to_csv(
    'work/outputs/action_playbook_queue.csv',
    index=False
)

# Evaluation file: includes the observed target for analysis only.
evaluation_cols = queue_cols + [
    'is_declining_label'
]

queue[evaluation_cols].to_csv(
    'work/outputs/action_playbook_evaluation.csv',
    index=False
)

summary = {
    'base_rate': float(base_rate),
    'n_scored': int(len(queue)),
    'n_clients': int(n_clients),
    'cv_n_splits': int(gkf.get_n_splits()),
    'n_disagreements_with_baseline': int(
        len(disagreement)
    ),
    'precision_at_20': float(
        precision_at_k(
            y,
            model_data['risk_score'],
            20
        )
    ),
    'precision_at_50': float(
        precision_at_k(
            y,
            model_data['risk_score'],
            50
        )
    ),
    'precision_at_100': float(
        precision_at_k(
            y,
            model_data['risk_score'],
            100
        )
    ),
    'precision_at_200': float(
        precision_at_k(
            y,
            model_data['risk_score'],
            200
        )
    )
}

with open(
    'work/outputs/playbook_summary.json',
    'w'
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(
    'Wrote work/outputs/action_playbook_queue.csv, '
    'action_playbook_evaluation.csv and '
    'playbook_summary.json'
)

print(summary)

Wrote work/outputs/action_playbook_queue.csv, action_playbook_evaluation.csv and playbook_summary.json
{'base_rate': 0.5420666666666667, 'n_scored': 30000, 'n_clients': 32, 'cv_n_splits': 5, 'n_disagreements_with_baseline': 15481, 'precision_at_20': 0.95, 'precision_at_50': 0.92, 'precision_at_100': 0.94, 'precision_at_200': 0.855}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.